# Workshop 8: Linear Regression — From Math to Implementation

Welcome to Workshop 8! So far you have explored data preprocessing, feature engineering, and exploratory data analysis. Today we take the first major step into **supervised learning** by studying **Linear Regression** — one of the oldest, most interpretable, and most widely used algorithms in machine learning.

Linear regression is used whenever you need to predict a **continuous numeric output** from one or more input features: predicting house prices, estimating energy consumption, forecasting sales, or modelling the relationship between dose and response in pharmacology.

## Learning Objectives

By the end of this workshop you will be able to:

1. **Derive** the mathematical formulation of linear regression, including the cost function and its gradients.
2. **Implement** simple linear regression from scratch using NumPy and the Normal Equation.
3. **Use** scikit-learn's `LinearRegression` for both simple and multiple regression.
4. **Evaluate** a regression model using MSE, RMSE, MAE, and R², and interpret what each metric tells you.
5. **Diagnose** model behaviour through residual plots, predicted-vs-actual plots, and learning curves.


## 1. Theory: What is Linear Regression?

### 1.1 The Model

Given an input vector $\mathbf{x} \in \mathbb{R}^d$ of $d$ features, linear regression models the output $y$ as a **linear combination** of the features plus a bias term:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b = \mathbf{w}^\top \mathbf{x} + b$$

where $\mathbf{w} = (w_1, \ldots, w_d)^\top$ is the **weight vector** and $b$ is the **bias** (intercept).

### 1.2 Matrix Form

For a dataset of $n$ samples, it is convenient to absorb the bias into the weight vector by augmenting each sample with a constant feature equal to 1. Define the **design matrix**:

$$\mathbf{X} = \begin{pmatrix} 1 & x_{11} & \cdots & x_{1d} \\ 1 & x_{21} & \cdots & x_{2d} \\ \vdots & \vdots & & \vdots \\ 1 & x_{n1} & \cdots & x_{nd} \end{pmatrix} \in \mathbb{R}^{n \times (d+1)}$$

and the augmented weight vector $\tilde{\mathbf{w}} = (b, w_1, \ldots, w_d)^\top$. Then all predictions are computed in a single matrix-vector product:

$$\hat{\mathbf{y}} = \mathbf{X}\tilde{\mathbf{w}}$$

### 1.3 Geometric Intuition

In the simple case ($d = 1$) the model is a **straight line** in 2D. In higher dimensions it becomes a **hyperplane** in $(d+1)$-dimensional space. Learning the model means finding the hyperplane that is *closest* to all the training points simultaneously — where "closest" will be defined precisely by the cost function in the next section.

The bias $b$ controls the vertical shift of the hyperplane, while each weight $w_i$ controls how much the prediction changes when feature $x_i$ increases by one unit, keeping all other features fixed.


## 2. Theory: The Cost Function

### 2.1 Mean Squared Error

To measure how well our model fits the training data we need a **cost function** (also called a loss function). The standard choice for regression is the **Mean Squared Error (MSE)**:

$$\mathcal{L}(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} \left( \hat{y}^{(i)} - y^{(i)} \right)^2 = \frac{1}{n} \| \mathbf{X}\tilde{\mathbf{w}} - \mathbf{y} \|^2$$

### 2.2 Why MSE?

- **Differentiable everywhere**: the squared function has a smooth gradient, which makes optimisation straightforward.
- **Convex**: the MSE loss surface is a paraboloid — it has a single global minimum, so any gradient-based optimiser will converge to the optimal solution.
- **Penalises large errors heavily**: squaring the residuals means that outlier predictions are penalised much more than small errors, encouraging the model to reduce large mistakes.
- **Probabilistic justification**: if the noise is Gaussian, minimising MSE is equivalent to Maximum Likelihood Estimation.

### 2.3 The Optimisation Problem

We want to find the parameters that minimise the cost:

$$\tilde{\mathbf{w}}^* = \arg\min_{\tilde{\mathbf{w}}} \; \frac{1}{n} \| \mathbf{X}\tilde{\mathbf{w}} - \mathbf{y} \|^2$$

There are two classical ways to solve this: **Gradient Descent** (iterative, scales to large datasets) and the **Normal Equation** (closed-form, exact but expensive for many features).


## 3. Theory: Gradient Descent

### 3.1 Partial Derivatives

To minimise $\mathcal{L}$ we follow the direction of steepest descent in parameter space. Computing partial derivatives:

$$\frac{\partial \mathcal{L}}{\partial w_j} = \frac{2}{n} \sum_{i=1}^{n} \left( \hat{y}^{(i)} - y^{(i)} \right) x_j^{(i)}$$

$$\frac{\partial \mathcal{L}}{\partial b} = \frac{2}{n} \sum_{i=1}^{n} \left( \hat{y}^{(i)} - y^{(i)} \right)$$

In matrix form, the full gradient with respect to the augmented weight vector is:

$$\nabla_{\tilde{\mathbf{w}}} \mathcal{L} = \frac{2}{n} \mathbf{X}^\top (\mathbf{X}\tilde{\mathbf{w}} - \mathbf{y})$$

### 3.2 Update Rules

At each iteration $t$, we update all parameters simultaneously:

$$w_j \leftarrow w_j - \alpha \frac{\partial \mathcal{L}}{\partial w_j}$$

$$b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

where $\alpha > 0$ is the **learning rate**.

### 3.3 Learning Rate Intuition

- **Too large**: the algorithm overshoots the minimum and may diverge.
- **Too small**: the algorithm converges, but very slowly, requiring many iterations.
- **Just right**: fast and stable convergence to the minimum.

In practice, common starting values are $\alpha \in \{0.01, 0.001, 0.1\}$. Feature scaling (e.g. standardisation) is essential before applying gradient descent, because features on different scales cause elongated loss surfaces where gradient descent oscillates.

### 3.4 Convergence

For linear regression with MSE, the loss surface is **strictly convex**, so gradient descent with a small enough learning rate is guaranteed to converge to the unique global minimum. A common stopping criterion is: stop when the change in loss between iterations is smaller than a threshold $\varepsilon$ (e.g. $10^{-6}$).


## 4. Theory: The Normal Equation

### 4.1 Closed-Form Solution

Because the MSE cost function is convex and differentiable, we can set the gradient to zero and solve analytically. Setting $\nabla_{\tilde{\mathbf{w}}} \mathcal{L} = 0$:

$$\mathbf{X}^\top (\mathbf{X}\tilde{\mathbf{w}} - \mathbf{y}) = 0$$

$$\mathbf{X}^\top \mathbf{X} \, \tilde{\mathbf{w}} = \mathbf{X}^\top \mathbf{y}$$

$$\boxed{\tilde{\mathbf{w}}^* = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}}$$

This is called the **Normal Equation**. It gives the exact optimal weights in a single computation — no iterations, no learning rate to tune.

### 4.2 Normal Equation vs Gradient Descent

| | Normal Equation | Gradient Descent |
|---|---|---|
| **Hyperparameters** | None | Learning rate $\alpha$, iterations |
| **Complexity** | $O(d^3)$ for matrix inverse | $O(n \cdot d)$ per iteration |
| **Scales with $n$** | Yes — but OK for moderate $n$ | Very well (mini-batch) |
| **Scales with $d$** | Poor for $d > 10^4$ | Well |
| **Feature scaling** | Not required | Required |
| **Exact solution** | Yes | Approximate (depends on iterations) |

**Rule of thumb**: use the Normal Equation when $d < 10{,}000$ and $n$ is not huge; prefer gradient descent (or its stochastic variants) for large-scale problems.

> **Note**: If $\mathbf{X}^\top \mathbf{X}$ is singular (non-invertible), use the Moore-Penrose pseudo-inverse $\mathbf{X}^+$ instead. This happens when features are linearly dependent or when $n < d$.


In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import scipy.stats as stats

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("All imports successful.")


In [ ]:
# --- Load California Housing Dataset ---
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("=" * 55)
print("CALIFORNIA HOUSING DATASET")
print("=" * 55)
print(f"Shape          : {df.shape}")
print(f"Target column  : MedHouseVal (median house value, $100k)")
print()
print("Feature List:")
for name, desc in zip(housing.feature_names, housing.feature_names):
    print(f"  {name}")

print()
print("--- Basic Statistics ---")
display(df.describe().round(2))

# --- Correlation heatmap with target ---
fig, ax = plt.subplots(figsize=(9, 6))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Correlation Matrix — California Housing", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print("Correlations with MedHouseVal (target):")
print(corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False).to_string())


In [ ]:
# --- Feature Selection & Preprocessing ---

# We start with the single most correlated feature for visualisation purposes
SIMPLE_FEATURE = 'MedInc'   # Median income — strongest linear relationship with price
TARGET = 'MedHouseVal'

# --- Simple regression (1 feature) ---
X_simple = df[[SIMPLE_FEATURE]].values   # shape (n, 1)
y = df[TARGET].values                     # shape (n,)

# Train / test split — 80 / 20
X_train_s, X_test_s, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

# Standardise features (mean=0, std=1)
scaler_s = StandardScaler()
X_train_s_scaled = scaler_s.fit_transform(X_train_s)
X_test_s_scaled  = scaler_s.transform(X_test_s)

print("Simple regression setup")
print(f"  Training samples : {X_train_s_scaled.shape[0]}")
print(f"  Test samples     : {X_test_s_scaled.shape[0]}")
print(f"  Feature          : {SIMPLE_FEATURE}")
print()

# --- Multiple regression (all features) ---
X_all = df[housing.feature_names].values   # shape (n, 8)

X_train_m, X_test_m, _, _ = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

scaler_m = StandardScaler()
X_train_m_scaled = scaler_m.fit_transform(X_train_m)
X_test_m_scaled  = scaler_m.transform(X_test_m)

print("Multiple regression setup")
print(f"  Training samples : {X_train_m_scaled.shape[0]}")
print(f"  Test samples     : {X_test_m_scaled.shape[0]}")
print(f"  Features         : {housing.feature_names}")


## 5. Simple Linear Regression (1 Feature)

Before jumping to 8 features, we start with a **single feature** — median income (`MedInc`) — for two reasons:

1. **Visualisation**: with one feature we can plot the data and the fitted line in 2D, making it easy to build intuition.
2. **Validation**: the closed-form solution simplifies to a scalar problem, so we can verify our implementation easily.

From the correlation matrix we saw that `MedInc` has the strongest linear relationship with the target (`MedHouseVal`, $r \approx 0.69$), making it the best single predictor.


In [ ]:
class SimpleLinearRegression:
    """
    Simple (and multiple) linear regression via the Normal Equation.

    The model: y_hat = X @ w  where X is already augmented with a column of ones
    to absorb the bias term.

    Normal Equation solution: w* = (X^T X)^{-1} X^T y
    We use np.linalg.lstsq for numerical stability (handles near-singular matrices).
    """

    def __init__(self):
        self.weights = None   # shape (d+1,) — first element is bias

    # ------------------------------------------------------------------
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'SimpleLinearRegression':
        """
        Fit the model using the Normal Equation.

        Parameters
        ----------
        X : array of shape (n_samples, n_features)
        y : array of shape (n_samples,)

        Returns
        -------
        self
        """
        n = X.shape[0]

        # Step 1: augment X with a column of ones for the bias term
        #         X_aug shape: (n, d+1)
        ones = np.ones((n, 1))
        X_aug = np.hstack([ones, X])        # bias column first

        # Step 2: solve the Normal Equation via least squares
        #         np.linalg.lstsq solves: min_w ||X_aug @ w - y||^2
        #         and is numerically more stable than explicit matrix inversion
        self.weights, _, _, _ = np.linalg.lstsq(X_aug, y, rcond=None)

        return self

    # ------------------------------------------------------------------
    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Predict continuous targets for input X.

        Parameters
        ----------
        X : array of shape (n_samples, n_features)

        Returns
        -------
        y_pred : array of shape (n_samples,)
        """
        if self.weights is None:
            raise RuntimeError("Call fit() before predict().")

        # Augment with bias column
        ones = np.ones((X.shape[0], 1))
        X_aug = np.hstack([ones, X])

        # Linear prediction: X_aug @ w
        return X_aug @ self.weights

    # ------------------------------------------------------------------
    @staticmethod
    def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Mean Squared Error: (1/n) * sum((y_pred - y_true)^2)
        """
        residuals = y_pred - y_true
        return float(np.mean(residuals ** 2))

    # ------------------------------------------------------------------
    @property
    def intercept_(self):
        """Return bias term (first element of weights)."""
        return self.weights[0] if self.weights is not None else None

    @property
    def coef_(self):
        """Return feature weights (all elements except bias)."""
        return self.weights[1:] if self.weights is not None else None


print("SimpleLinearRegression class defined.")


In [ ]:
# --- Fit our scratch model ---
scratch_model = SimpleLinearRegression()
scratch_model.fit(X_train_s_scaled, y_train)

y_pred_scratch_train = scratch_model.predict(X_train_s_scaled)
y_pred_scratch_test  = scratch_model.predict(X_test_s_scaled)

# --- Fit sklearn model ---
sklearn_model = LinearRegression()
sklearn_model.fit(X_train_s_scaled, y_train)

y_pred_sklearn_train = sklearn_model.predict(X_train_s_scaled)
y_pred_sklearn_test  = sklearn_model.predict(X_test_s_scaled)

# --- Compare coefficients ---
print("=" * 50)
print("COEFFICIENT COMPARISON (Simple Linear Regression)")
print("=" * 50)
print(f"{'Parameter':<15} {'Scratch':>12} {'Sklearn':>12}")
print("-" * 40)
print(f"{'Intercept (b)':<15} {scratch_model.intercept_:>12.6f} {sklearn_model.intercept_:>12.6f}")
print(f"{'Weight (w)':<15} {scratch_model.coef_[0]:>12.6f} {sklearn_model.coef_[0]:>12.6f}")
print()

# --- Compare MSE ---
mse_scratch = scratch_model.mse(y_test, y_pred_scratch_test)
mse_sklearn = mean_squared_error(y_test, y_pred_sklearn_test)
print(f"Test MSE — Scratch : {mse_scratch:.6f}")
print(f"Test MSE — Sklearn : {mse_sklearn:.6f}")
print()
print("Both implementations agree: the Normal Equation yields the same solution.")


In [ ]:
# --- Visualisation 1: Regression Line ---
fig, ax = plt.subplots(figsize=(9, 6))

# Scatter plot: test set points
ax.scatter(
    X_test_s_scaled, y_test,
    alpha=0.3, s=15, color='steelblue', label='Test samples'
)

# Regression line: evaluate on a fine grid
x_line = np.linspace(X_test_s_scaled.min(), X_test_s_scaled.max(), 300).reshape(-1, 1)
y_line = scratch_model.predict(x_line)
ax.plot(x_line, y_line, color='crimson', linewidth=2.5, label='Regression line (scratch)')

ax.set_xlabel('MedInc (standardised)', fontsize=12)
ax.set_ylabel('Median House Value ($100k)', fontsize=12)
ax.set_title(
    f'Simple Linear Regression: MedInc → MedHouseVal\n'
    f'$\\hat{{y}} = {scratch_model.coef_[0]:.3f}\\, x + {scratch_model.intercept_:.3f}$',
    fontsize=13
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# --- Visualisation 2: Residual Plot ---
residuals_test = y_pred_scratch_test - y_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: residuals vs predicted values
axes[0].scatter(y_pred_scratch_test, residuals_test, alpha=0.3, s=15, color='darkorange')
axes[0].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Predicted Value ($\\hat{y}$)', fontsize=12)
axes[0].set_ylabel('Residual ($\\hat{y} - y$)', fontsize=12)
axes[0].set_title('Residuals vs Predicted Values', fontsize=13)

# Right: distribution of residuals
axes[1].hist(residuals_test, bins=60, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Residual', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Residuals', fontsize=13)

plt.suptitle('Residual Diagnostics — Simple Linear Regression', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - A GOOD residual plot shows points scattered randomly around the horizontal zero line.")
print("  - Patterns (e.g. a funnel shape, curve) indicate model mis-specification.")
print("  - The clipping visible near y=5 suggests the dataset caps house values at $500k.")


In [ ]:
# --- Multiple Linear Regression: all 8 features ---
multi_model = LinearRegression()
multi_model.fit(X_train_m_scaled, y_train)

y_pred_multi_train = multi_model.predict(X_train_m_scaled)
y_pred_multi_test  = multi_model.predict(X_test_m_scaled)

# --- Coefficient table ---
coef_df = pd.DataFrame({
    'Feature'    : housing.feature_names,
    'Coefficient': multi_model.coef_
}).sort_values('Coefficient', ascending=False)

print("=" * 50)
print("MULTIPLE LINEAR REGRESSION — Coefficients")
print(f"Intercept (b) = {multi_model.intercept_:.4f}")
print("=" * 50)
print(coef_df.to_string(index=False))
print()
print("Note: coefficients are on the standardised feature scale.")
print("A positive coefficient means the feature is positively associated with house value.")

# Bar chart of coefficients
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Coefficient value (standardised features)', fontsize=12)
ax.set_title('Feature Coefficients — Multiple Linear Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# --- Model Evaluation: Multiple Regression ---

def evaluation_report(y_true_train, y_pred_train, y_true_test, y_pred_test, model_name='Model'):
    """Print a formatted evaluation table for train and test sets."""
    metrics = {
        'MSE' : (mean_squared_error,  {}),
        'RMSE': (mean_squared_error,  {}),
        'MAE' : (mean_absolute_error, {}),
        'R²'  : (r2_score,            {}),
    }

    print(f"\n{'=' * 52}")
    print(f"  {model_name}")
    print(f"{'=' * 52}")
    print(f"  {'Metric':<8} {'Train':>12} {'Test':>12}")
    print(f"  {'-' * 34}")

    for name, (fn, kwargs) in metrics.items():
        if name == 'RMSE':
            train_val = np.sqrt(fn(y_true_train, y_pred_train, **kwargs))
            test_val  = np.sqrt(fn(y_true_test,  y_pred_test,  **kwargs))
        else:
            train_val = fn(y_true_train, y_pred_train, **kwargs)
            test_val  = fn(y_true_test,  y_pred_test,  **kwargs)
        print(f"  {name:<8} {train_val:>12.4f} {test_val:>12.4f}")

    print(f"{'=' * 52}")


evaluation_report(
    y_train, y_pred_scratch_train,
    y_test,  y_pred_scratch_test,
    model_name="Simple LR — 1 feature (MedInc) — Scratch"
)

evaluation_report(
    y_train, y_pred_multi_train,
    y_test,  y_pred_multi_test,
    model_name="Multiple LR — 8 features — Sklearn"
)


## 6. Theory: Evaluation Metrics

### 6.1 Mean Squared Error (MSE) and RMSE

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}^{(i)} - y^{(i)})^2$$

$$\text{RMSE} = \sqrt{\text{MSE}}$$

**RMSE** is in the same unit as the target variable (here: $100k USD), making it more interpretable than MSE. An RMSE of 0.73 means our predictions are off by $73,000 on average (in the root-mean-square sense).

### 6.2 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |\hat{y}^{(i)} - y^{(i)}|$$

MAE is **more robust to outliers** than RMSE because it does not square the residuals. If you care more about typical errors than about avoiding catastrophic predictions, MAE is the better metric.

### 6.3 R² — Coefficient of Determination

$$R^2 = 1 - \frac{\sum_{i=1}^{n}(\hat{y}^{(i)} - y^{(i)})^2}{\sum_{i=1}^{n}(\bar{y} - y^{(i)})^2} = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}$$

**Interpretation**:
- $R^2 = 1$: the model explains all variance in the target — perfect predictions.
- $R^2 = 0$: the model is no better than always predicting the mean $\bar{y}$.
- $R^2 < 0$: the model is *worse* than predicting the mean (possible on the test set).
- **$R^2 = 0.6$** means the model explains **60%** of the total variance in $y$. The remaining 40% is either noise, non-linear structure, or information not captured by the chosen features.

### 6.4 Which Metric to Report?

Always report **at least two metrics**. A common choice is RMSE (penalises large errors) and $R^2$ (scale-free, easy to communicate). Report both train and test values to detect overfitting.


In [ ]:
# --- Visualisation 3: Predicted vs Actual ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (y_pred, title) in zip(axes, [
    (y_pred_scratch_test, 'Simple LR (1 feature)'),
    (y_pred_multi_test,   'Multiple LR (8 features)'),
]):
    r2 = r2_score(y_test, y_pred)

    ax.scatter(y_test, y_pred, alpha=0.25, s=12, color='steelblue')

    # Ideal line: y_pred == y_true
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Ideal (y = ŷ)')

    ax.set_xlabel('Actual Value ($100k)', fontsize=12)
    ax.set_ylabel('Predicted Value ($100k)', fontsize=12)
    ax.set_title(f'{title}\n$R^2 = {r2:.3f}$', fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.suptitle('Predicted vs Actual — Test Set', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Interpretation: points on the red dashed line are perfect predictions.")
print("The tighter the cloud around this line, the better the model.")


In [ ]:
# --- Visualisation 4: Learning Curve  ---
# A learning curve shows how performance changes as we add more training data.
# It helps diagnose bias (underfitting) vs variance (overfitting).

train_sizes, train_scores, val_scores = learning_curve(
    LinearRegression(),
    X_train_m_scaled, y_train,
    train_sizes=np.linspace(0.05, 1.0, 20),
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

# Convert negative MSE to positive RMSE
train_rmse = np.sqrt(-train_scores)
val_rmse   = np.sqrt(-val_scores)

train_mean = train_rmse.mean(axis=1)
train_std  = train_rmse.std(axis=1)
val_mean   = val_rmse.mean(axis=1)
val_std    = val_rmse.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Training RMSE')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='steelblue')

ax.plot(train_sizes, val_mean, 'o-', color='tomato', label='Validation RMSE (CV)')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='tomato')

ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('Learning Curve — Multiple Linear Regression', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Both curves converge → the model has HIGH BIAS (underfitting).")
print("    Adding more data will not help much — we need richer features or a different model.")
print("  - Large gap between curves → HIGH VARIANCE (overfitting).")
print("    More data or regularisation would help close the gap.")


## 7. Theory: Assumptions of Linear Regression

Linear regression is a powerful tool, but it relies on four key assumptions. Violating them does not mean the model is useless, but the estimates may be biased, inefficient, or the standard errors invalid.

### Assumption 1 — Linearity
The relationship between each feature and the target must be **linear** (possibly after transformations). If the true relationship is curved, the model will systematically under- or over-predict. *Check*: residuals vs fitted plot — look for curvature.

### Assumption 2 — Independence of Errors
The residuals $\varepsilon^{(i)} = y^{(i)} - \hat{y}^{(i)}$ must be **independent** of each other. This is often violated in time series data (autocorrelation) or spatial data (spatial correlation). *Check*: Durbin-Watson statistic for time series.

### Assumption 3 — Homoscedasticity
The variance of the residuals must be **constant** across all levels of the fitted values — i.e., $\text{Var}(\varepsilon | \mathbf{x}) = \sigma^2$. If the residuals fan out as $\hat{y}$ increases, the assumption is violated (heteroscedasticity). *Check*: residual vs fitted plot — look for a funnel shape.

### Assumption 4 — Normality of Residuals
For valid statistical inference (confidence intervals, p-values), the residuals should be **approximately normally distributed**: $\varepsilon \sim \mathcal{N}(0, \sigma^2)$. This is *not* required for point predictions to be valid (only for inference). *Check*: histogram of residuals, Q-Q plot.


In [ ]:
# --- Check Normality of Residuals (Multiple Regression) ---
residuals_multi = y_pred_multi_test - y_test

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Histogram of residuals
axes[0].hist(residuals_multi, bins=80, color='mediumpurple', edgecolor='white', alpha=0.8, density=True)

# Overlay a normal distribution for comparison
mu, sigma = residuals_multi.mean(), residuals_multi.std()
x_norm = np.linspace(residuals_multi.min(), residuals_multi.max(), 300)
axes[0].plot(x_norm, stats.norm.pdf(x_norm, mu, sigma),
             'r-', linewidth=2, label=f'$\\mathcal{{N}}({mu:.2f},\\, {sigma:.2f}^2)$')
axes[0].axvline(0, color='black', linewidth=1.2, linestyle='--')
axes[0].set_xlabel('Residual', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Histogram of Residuals', fontsize=13)
axes[0].legend(fontsize=11)

# Right: Q-Q plot
stats.probplot(residuals_multi, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals', fontsize=13)
axes[1].get_lines()[1].set_color('crimson')   # reference line

plt.suptitle('Normality Check — Multiple Linear Regression Residuals',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Shapiro-Wilk test on a subsample (full test set may be too large)
sample = np.random.choice(residuals_multi, size=min(500, len(residuals_multi)), replace=False)
stat, p_value = stats.shapiro(sample)
print(f"Shapiro-Wilk test (n=500 subsample): W={stat:.4f}, p={p_value:.4e}")
print("A small p-value (< 0.05) suggests the residuals are NOT perfectly normal.")
print("For large datasets this test is very sensitive — rely on the Q-Q plot for practical assessment.")


## 8. Exercises

The following exercises are designed to deepen your understanding. Work through them in order — each one builds on the previous.

---

### Exercise 1 — Polynomial Features

Linear regression can only capture linear relationships. One way to extend it is to add **polynomial features**: instead of using $x$ as input, we use $(x, x^2, x^3, \ldots)$. The model is still *linear in the parameters*, so all the theory above still applies.

**Task**: Use `sklearn.preprocessing.PolynomialFeatures` with `degree=2` on the full feature set. Fit a linear regression on the polynomial-expanded features and compare the test $R^2$ with the plain multiple regression from Cell 14.

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('lr', LinearRegression()),
])
# TODO: fit on (X_train_m, y_train), evaluate on (X_test_m, y_test)
# Hint: use the un-scaled X_train_m / X_test_m as input — the pipeline will scale internally
```

**Questions**:
- Does $R^2$ improve? By how much?
- How many features does degree-2 expansion produce from 8 original features? (Formula: $\binom{d+p}{p}$)
- What is the risk of going to degree 3 or higher?

---

### Exercise 2 — Remove Price-Capped Outliers

In the residual plot (Cell 13) you may have noticed a horizontal band of points at `MedHouseVal = 5.0`. This is because the dataset **caps** house values at $\$500\,000$. These censored observations can distort the regression.

**Task**: Remove all samples where `MedHouseVal >= 4.5`, retrain the multiple regression model, and compare RMSE and $R^2$ with the original model.

```python
df_clean = df[df['MedHouseVal'] < 4.5].copy()
# TODO: split, scale, train, and evaluate
```

**Questions**:
- Does performance improve on the cleaned dataset?
- Is it fair to remove these points? In what real-world scenarios would this be (un)acceptable?
- What fraction of the data is removed?

---

### Exercise 3 — Gradient Descent from Scratch

Implement **batch gradient descent** from scratch and verify that it converges to the same solution as the Normal Equation.

```python
def gradient_descent_lr(X, y, alpha=0.01, n_iter=1000):
    """
    Batch gradient descent for linear regression.

    Parameters
    ----------
    X      : array (n, d) — already standardised, WITHOUT bias column
    y      : array (n,)
    alpha  : learning rate
    n_iter : number of iterations

    Returns
    -------
    w : array (d,) — feature weights
    b : float      — bias
    history : list of MSE values per iteration
    """
    n, d = X.shape
    w = np.zeros(d)   # initialise weights to zero
    b = 0.0           # initialise bias to zero
    history = []

    for _ in range(n_iter):
        y_hat = X @ w + b
        residual = y_hat - y

        # TODO: compute gradients dw and db
        # TODO: update w and b
        # TODO: append MSE to history

    return w, b, history

# After implementing, run:
# w_gd, b_gd, history = gradient_descent_lr(X_train_m_scaled, y_train, alpha=0.1, n_iter=500)
# plt.plot(history); plt.xlabel('Iteration'); plt.ylabel('MSE'); plt.title('GD Convergence')
```

**Questions**:
- Plot the loss curve. How many iterations does it take to converge?
- Try `alpha = 1.0`. What happens?
- Compare the final weights with those from `sklearn.LinearRegression`. Do they match?


## Summary

### Key Takeaways

In this workshop we covered Linear Regression end-to-end:

| Topic | Key Result |
|---|---|
| **Model** | $\hat{y} = \mathbf{w}^\top \mathbf{x} + b$ — a weighted sum of features |
| **Cost** | MSE — convex, differentiable, probabilistically motivated |
| **Normal Equation** | $\tilde{\mathbf{w}}^* = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$ — exact, $O(d^3)$ |
| **Gradient Descent** | Iterative, scalable, requires feature scaling and learning-rate tuning |
| **Evaluation** | RMSE (interpretable), MAE (robust), $R^2$ (scale-free) |
| **Diagnostics** | Residual plot, Q-Q plot, learning curve |
| **Assumptions** | Linearity, independence, homoscedasticity, normality of residuals |

On the California Housing dataset we found:
- Simple LR (MedInc only): $R^2 \approx 0.47$ — income alone is a decent but incomplete predictor.
- Multiple LR (8 features): $R^2 \approx 0.60$ — all features together capture 60% of the variance.
- The learning curve suggests the model is **underfitting** — it has high bias. This motivates the exercises (polynomial features) and upcoming workshops.

### What's Next?

**Workshop 9 — Logistic Regression**: we move from continuous to **binary** output. The linear model $\mathbf{w}^\top \mathbf{x} + b$ is wrapped inside a sigmoid function $\sigma(z) = \frac{1}{1 + e^{-z}}$ to produce probabilities, and we minimise the **cross-entropy** loss instead of MSE. The mathematical machinery — gradients, Normal Equation analogue, evaluation metrics — all evolve in natural ways. See you there!
